<a href="https://colab.research.google.com/github/brindatenn/phd-prep-summer-2026/blob/main/week1/pytorch_workflow_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch Workflow Fundamentals

**The core idea:** take past data → build a model to find patterns in it → use those patterns to predict.
Here we fit a straight line (`y = weight * X + bias`) with linear regression as the running example.

### The standard PyTorch workflow
1. **Get data ready** (turn it into tensors)
2. **Build a model** (+ pick loss function & optimizer)
3. **Fit the model to data** (training loop)
4. **Make predictions & evaluate** (inference)
5. **Save & load** the model
6. **Put it all together** (device-agnostic)


## Setup

In [ ]:
import torch
from torch import nn          # nn = all of PyTorch's neural network building blocks
import matplotlib.pyplot as plt

# Device-agnostic code: use GPU if available, else CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__} | Using device: {device}")

## 1. Data - prepare and load

ML is a game of two parts: **(1)** turn data into numbers, **(2)** build a model to learn the representation.

We create data with *known* parameters (`weight`, `bias`) so we can later check how close the model gets.


In [ ]:
# Known parameters (the model will try to learn these)
weight = 0.7
bias = 0.3

# Create a straight line: y = weight * X + bias
start, end, step = 0, 1, 0.02
X = torch.arange(start, end, step).unsqueeze(dim=1)  # unsqueeze -> shape [N, 1]; avoids shape errors in linear layers
y = weight * X + bias

X[:5], y[:5]

### Train / test split

| Split | Purpose | ~% of data |
|---|---|---|
| **Training** | Model learns from this | 60–80% |
| **Validation** (optional) | Tune the model | 10–20% |
| **Testing** | Evaluate generalization to unseen data | 10–20% |

> Keep the test set separate - it's how you check the model generalizes.


In [ ]:
# 80/20 split (no shuffling needed here since data is ordered & simple)
train_split = int(0.8 * len(X))
X_train, y_train = X[:train_split], y[:train_split]
X_test,  y_test  = X[train_split:], y[train_split:]

len(X_train), len(y_train), len(X_test), len(y_test)

### Visualize, visualize, visualize
A reusable helper to plot training data, test data, and predictions.

In [ ]:
def plot_predictions(train_data=X_train, train_labels=y_train,
                     test_data=X_test, test_labels=y_test,
                     predictions=None):
    """Plots training/test data and (optionally) predictions."""
    plt.figure(figsize=(10, 7))
    plt.scatter(train_data, train_labels, c="b", s=4, label="Training data")
    plt.scatter(test_data,  test_labels,  c="g", s=4, label="Testing data")
    if predictions is not None:
        plt.scatter(test_data, predictions, c="r", s=4, label="Predictions")
    plt.legend(prop={"size": 14})

plot_predictions()

## 2. Build a model

**PyTorch model-building essentials:**

| Module | What it does |
|---|---|
| `torch.nn` | Building blocks for computational graphs (networks) |
| `nn.Module` | Base class for all models — subclass it, and **you must define `forward()`** |
| `nn.Parameter` | Stores learnable tensors; `requires_grad=True` → gradients tracked (autograd) |
| `torch.optim` | Optimization algorithms that update parameters to reduce loss |
| `forward()` | Defines the computation the model performs on input data |

Version A builds the parameters **manually** with `nn.Parameter` to show what's happening under the hood.


In [ ]:
class LinearRegressionModel(nn.Module):     # almost everything in PyTorch subclasses nn.Module
    def __init__(self):
        super().__init__()
        # Start with random weight & bias; learned via gradient descent
        self.weights = nn.Parameter(torch.randn(1, dtype=torch.float), requires_grad=True)
        self.bias    = nn.Parameter(torch.randn(1, dtype=torch.float), requires_grad=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.weights * x + self.bias   # linear regression: y = m*x + b

torch.manual_seed(42)          # nn.Parameter is randomly initialized -> seed for reproducibility
model_0 = LinearRegressionModel()

list(model_0.parameters())     # the learnable parameters

`.state_dict()` shows the model's named parameters (its current state):

In [ ]:
model_0.state_dict()

### Making predictions with `torch.inference_mode()`

Data passed to the model flows through `forward()`. Wrap inference in `torch.inference_mode()` - it turns off gradient tracking (not needed for prediction) so forward passes are faster.
> Older code uses `torch.no_grad()`; `inference_mode()` is the newer, preferred version.


In [ ]:
# Untrained model -> predictions will be poor (random parameters)
with torch.inference_mode():
    y_preds = model_0(X_test)

plot_predictions(predictions=y_preds)

## 3. Train the model

To improve the random parameters we need two things:

| Thing | Role | Common choice |
|---|---|---|
| **Loss function** | Measures how *wrong* predictions are (lower = better) | `nn.L1Loss()` = MAE (regression); `nn.BCELoss()` (binary classification) |
| **Optimizer** | Tells the model *how* to update its parameters to reduce loss | `torch.optim.SGD(params, lr)`, `torch.optim.Adam(...)` |

**Learning rate (`lr`)** is a hyperparameter: too high = unstable, too low = slow. Common starts: `0.01`, `0.001`.


In [ ]:
loss_fn = nn.L1Loss()                                   # MAE loss (a.k.a. L1)
optimizer = torch.optim.SGD(params=model_0.parameters(), # params to optimize
                            lr=0.01)                     # learning rate

### The PyTorch training & testing loop

**Training loop — the 5 core steps (memorize this pattern):**

| # | Step | Code |
|---|---|---|
| 1 | Forward pass | `y_pred = model(X_train)` |
| 2 | Calculate loss | `loss = loss_fn(y_pred, y_train)` |
| 3 | Zero gradients | `optimizer.zero_grad()` |
| 4 | Backpropagation | `loss.backward()` |
| 5 | Step optimizer (gradient descent) | `optimizer.step()` |

**Ordering rules of thumb:** loss *before* `backward()`; `zero_grad()` *before* `backward()`; `step()` *after* `backward()`.

**Testing loop** = forward pass + calculate loss only (no `backward()`, no `step()` — nothing is being learned).
An **epoch** = one pass through the training data.


In [ ]:
torch.manual_seed(42)
epochs = 100

for epoch in range(epochs):
    ### Training
    model_0.train()                       # training mode (default)
    y_pred = model_0(X_train)             # 1. forward pass
    loss = loss_fn(y_pred, y_train)       # 2. calculate loss
    optimizer.zero_grad()                 # 3. zero gradients (they accumulate by default)
    loss.backward()                       # 4. backpropagation
    optimizer.step()                      # 5. update parameters

    ### Testing
    model_0.eval()                        # evaluation mode
    with torch.inference_mode():
        test_pred = model_0(X_test)
        test_loss = loss_fn(test_pred, y_test)

    if epoch % 10 == 0:
        print(f"Epoch: {epoch} | Train loss: {loss:.4f} | Test loss: {test_loss:.4f}")

Check how close the learned parameters got to the true `weight=0.7`, `bias=0.3`:

In [ ]:
print("Learned:", model_0.state_dict())
print(f"Actual:  weight={weight}, bias={bias}")

## 4. Making predictions (inference)

**Three rules for inference with a PyTorch model:**
1. Put the model in eval mode: `model.eval()`
2. Use the inference context manager: `with torch.inference_mode():`
3. Model and data must be on the **same device** (both CPU or both GPU)


In [ ]:
model_0.eval()
with torch.inference_mode():
    y_preds = model_0(X_test)

plot_predictions(predictions=y_preds)

## 5. Saving & loading a model

Three key methods: `torch.save` (serialize with pickle), `torch.load` (deserialize),
`model.load_state_dict()` (load saved parameters into a model).

**Recommended:** save the `state_dict()` (just the learned parameters), not the whole model - saving the whole model binds it to the exact class/directory structure and breaks more easily.
Convention: filenames end in `.pth` or `.pt`.
> Only load models from sources you trust - `pickle` is not secure.


In [ ]:
from pathlib import Path

# 1. Create a models directory
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Build the save path
MODEL_SAVE_PATH = MODEL_PATH / "01_pytorch_workflow_model_0.pth"

# 3. Save the state_dict (learned parameters only)
torch.save(obj=model_0.state_dict(), f=MODEL_SAVE_PATH)
print(f"Saved to: {MODEL_SAVE_PATH}")

Load it back into a fresh instance and confirm predictions match:

In [ ]:
# New instance (random params) -> load saved params into it
loaded_model_0 = LinearRegressionModel()
loaded_model_0.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

loaded_model_0.eval()
with torch.inference_mode():
    loaded_preds = loaded_model_0(X_test)

y_preds == loaded_preds   # should all be True

## 6. Putting it together — `nn.Linear` + device-agnostic code

Instead of defining `weight`/`bias` by hand with `nn.Parameter`, use **`nn.Linear(in_features, out_features)`**
— it creates and manages the parameters for you. This is the more common, scalable approach.

- `in_features` = number of input dimensions, `out_features` = number of output dimensions (both `1` here).
- `.to(device)` moves models/tensors between CPU and GPU.


In [ ]:
class LinearRegressionModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_layer = nn.Linear(in_features=1, out_features=1)  # creates weight + bias for us

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear_layer(x)

torch.manual_seed(42)
model_1 = LinearRegressionModelV2()
model_1.to(device)                       # move model to GPU/CPU
model_1, model_1.state_dict()

Move data to the same device, then train (same 5-step loop, 1000 epochs):

In [ ]:
# Loss + optimizer for the new model
loss_fn = nn.L1Loss()
optimizer = torch.optim.SGD(params=model_1.parameters(), lr=0.01)

# Put data on the target device (model & data must match)
X_train, y_train = X_train.to(device), y_train.to(device)
X_test,  y_test  = X_test.to(device),  y_test.to(device)

torch.manual_seed(42)
epochs = 1000

for epoch in range(epochs):
    ### Training
    model_1.train()
    y_pred = model_1(X_train)             # 1. forward
    loss = loss_fn(y_pred, y_train)       # 2. loss
    optimizer.zero_grad()                 # 3. zero grad
    loss.backward()                       # 4. backprop
    optimizer.step()                      # 5. step

    ### Testing
    model_1.eval()
    with torch.inference_mode():
        test_pred = model_1(X_test)
        test_loss = loss_fn(test_pred, y_test)

    if epoch % 100 == 0:
        print(f"Epoch: {epoch} | Train loss: {loss:.5f} | Test loss: {test_loss:.5f}")

print("\nLearned:", model_1.state_dict())
print(f"Actual:  weight={weight}, bias={bias}")

Plotting note: libraries like matplotlib/NumPy can't use GPU tensors.
Call `.cpu()` on the tensor first.

In [ ]:
model_1.eval()
with torch.inference_mode():
    y_preds = model_1(X_test)

plot_predictions(predictions=y_preds.cpu())   # .cpu() so matplotlib can read it

### Save & load V2 (device-agnostic)

In [ ]:
MODEL_SAVE_PATH = Path("models") / "01_pytorch_workflow_model_1.pth"
torch.save(obj=model_1.state_dict(), f=MODEL_SAVE_PATH)

# Load into a fresh instance and send to device
loaded_model_1 = LinearRegressionModelV2()
loaded_model_1.load_state_dict(torch.load(MODEL_SAVE_PATH))
loaded_model_1.to(device)

loaded_model_1.eval()
with torch.inference_mode():
    loaded_preds = loaded_model_1(X_test)

y_preds == loaded_preds   # should all be True

## Quick recap

- **Data** → tensors → train/test split. Always visualize.
- **Model** → subclass `nn.Module`, define `forward()`. Use `nn.Parameter` (manual) or `nn.Linear` (preferred).
- **Loss + optimizer** → `nn.L1Loss()` (MAE) + `torch.optim.SGD(params, lr)` for regression.
- **Training loop (5 steps):** forward → loss → `zero_grad()` → `backward()` → `step()`.
- **Testing loop:** forward → loss only (no backprop).
- **Inference:** `model.eval()` + `torch.inference_mode()` + same device.
- **Save/load:** save `state_dict()` (`.pth`), load into a fresh model instance.
- **Device-agnostic:** `device = "cuda" if torch.cuda.is_available() else "cpu"`, then `.to(device)` on model & data.
